# 3D Scan Pipeline - Part 1: Data Preparation

**Objective:** Extract frames from video and run COLMAP Structure-from-Motion (SfM).
**Output:** A zip file `3d_scan_data_part1.zip` containing the sparse point cloud and images. You will upload this to Part 2.

**Environment:** Standard GPU (T4) or CPU (slower). No special constraints.

In [ ]:
import os

print("⏳ Setting up Environment (Part 1)...")

# 1. Clone Repo (Only if local project not found)
if not os.path.exists("3DSCAN"):
    !git clone https://github.com/PRIDA-TAKON/3DSCAN.git
    if os.path.exists("3DSCAN"):
        os.chdir("3DSCAN")
else:
    print("📂 Project folder found. Using local version.")
    if os.path.basename(os.getcwd()) != "3DSCAN" and os.path.exists("3DSCAN"):
        os.chdir("3DSCAN")

# 2. Install Dependencies for Data Prep
# Note: We don't need strict numpy constraints here, COLMAP relies on system binaries
print("⏳ Installing Dependencies...")
!pip install --upgrade pip
!pip install opencv-python opencv-python-headless opencv-contrib-python pandas tqdm

# Install COLMAP & FFmpeg
!apt-get update
!apt-get install -y colmap ffmpeg xvfb

print("✅ Part 1 Setup Complete.")

In [ ]:
print("=== STEP 1: Extract Frames ===")
# Find video automatically or set manually
import glob
video_path = None
search_paths = ["/kaggle/input", "input"]
for p in search_paths:
    found = glob.glob(f"{p}/**/*.mp4", recursive=True)
    if found:
        video_path = found[0]
        break

if not video_path:
    print("❌ No video found! Please upload a dataset.")
else:
    print(f"🎬 Found video: {video_path}")
    !python scripts/step1_extract_frames.py --input_video "{video_path}" --output_dir "working_data/3d_scan/images"

In [ ]:
print("=== STEP 2: COLMAP SfM ===")
if not os.path.exists("working_data/3d_scan/images"):
    print("❌ Images not found. Did Step 1 run?")
else:
    !python scripts/step2_colmap_sfm.py --images_dir "working_data/3d_scan/images" --output_dir "working_data/3d_scan"

In [ ]:
print("=== Compress Data for Part 2 ===")
output_zip = "3d_scan_data_part1.zip"

# Compress the working_data/3d_scan folder
# We need: sparse/0 (COLMAP model), images (extracted frames), transforms.json (if generated)
if os.path.exists("working_data/3d_scan"):
    !zip -r {output_zip} working_data/3d_scan
    
    from IPython.display import FileLink
    display(FileLink(output_zip))
    print(f"✅ Done. Please download '{output_zip}' and upload it to the Part 2 notebook.")
else:
    print("❌ No output data found to zip.")